# ITAI 1378 Lab 10: Video Analysis and Object Tracking

**Course:** ITAI 1378 Computer Vision and AI
**Module:** 10, Video Analysis and Object Tracking
**Points:** 100
**Estimated time:** 90 to 120 minutes
**Platform:** Google Colab (free tier is enough)

---

## What This Lab Is About

Every lab before this one worked on a single image at a time. In this lab you will
run a detector on a video, then add a tracker on top of it, and see with your own
eyes what identity actually means in computer vision.

You are not going to build a tracker from scratch. Nobody does that in 2026. You are
going to use Ultralytics YOLO with ByteTrack and BoT-SORT, which is exactly what a
junior CV engineer does on the job, and then you are going to look carefully at where
it breaks.

## Learning Objectives

By the end of this lab you will be able to:

1. Explain the difference between detection output and tracking output
2. Run tracking by detection on a real video using Ultralytics
3. Count unique track IDs and compare that number to the true object count
4. Recognize at least two tracking failure modes in your own output
5. Compare two trackers and explain the result using motion cues and appearance cues
6. Apply video tracking to a real Houston workplace task and name its privacy risks

## How To Use This Notebook

1. Open this notebook in Google Colab
2. Go to Runtime, then Change runtime type, then select T4 GPU, then Save
3. Run every cell in order from top to bottom. Do not skip cells
4. Wherever you see **YOUR ANSWER HERE**, replace that text with your own writing
5. Wherever you see `# TODO`, write the small piece of code that is asked for

Everything you need is in this notebook. You do not need any other file.

## What You Turn In

One file: this notebook, with all cells run and all outputs visible, saved as
`ITAI1378_Lab10_YourLastName.ipynb`. Download it from Colab with File, then
Download, then Download .ipynb.

Do not clear your outputs before you submit. The output is the evidence that you
ran the lab.

---
# Part 0: Setup

Run the two cells below. The first one installs the libraries. It takes about one
minute. You will see a lot of text scroll by, which is normal.

Ultralytics gives you YOLO detection and tracking in one package. OpenCV reads and
writes video files. Supervision gives us a free sample video to work with.

In [1]:
!pip install -q ultralytics supervision
print("Install finished.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 6.6 MB/s eta 0:00:00
Install finished.


In [2]:
import os, glob, collections
import cv2
import torch
from ultralytics import YOLO
from IPython.display import HTML
import base64

print("Torch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("No GPU found. The lab still works, it will just run slower.")
    print("To turn on the GPU: Runtime, Change runtime type, T4 GPU, Save.")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Torch version: 2.11.0+cu128
GPU available: True


### A helper for watching video inside the notebook

Colab cannot play every video format. The function below converts a video file to a
browser friendly format and displays it right in the notebook. You do not need to
understand this code. Just run the cell.

In [3]:
def show_video(path, width=640):
    """Convert a video to browser friendly h264 and display it in the notebook."""
    web_path = path.replace(".mp4", "_web.mp4").replace(".avi", "_web.mp4")
    os.system(f'ffmpeg -y -loglevel error -i "{path}" -vcodec libx264 "{web_path}"')
    data = open(web_path, "rb").read()
    b64 = base64.b64encode(data).decode()
    return HTML(f'<video width={width} controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>')

print("Helper ready.")

Helper ready.


---
# Part 1: Get a Video and Look at Its Numbers

We download a short sample video of people walking in a public square. Then we trim
it to 10 seconds. Ten seconds is plenty for this lab and it keeps every later cell
fast.

If you would rather use your own video, there are instructions at the end of Part 1.

In [4]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.PEOPLE_WALKING)
print("Downloaded:", os.path.exists("people-walking.mp4"))

# Trim to the first 10 seconds so every step in this lab runs quickly
!ffmpeg -y -loglevel error -i people-walking.mp4 -t 10 -c:v libx264 -an clip.mp4

VIDEO = "clip.mp4"
print("Working video:", VIDEO, os.path.getsize(VIDEO) // 1024, "KB")

[2026-08-03 01:25:25] [INFO] supervision.assets.downloader - Downloading people-walking.mp4 assets


  0%|          | 0/7606633 [00:00<?, ?it/s]

Downloaded: True
Working video: clip.mp4 4125 KB


### Read the video properties

Module 10 talked about frame rate, resolution, and bandwidth as real system design
constraints. Now measure them on an actual file.

In [5]:
cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS)
WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print("Frames per second:", round(FPS, 2))
print("Resolution:", WIDTH, "x", HEIGHT)
print("Total frames:", FRAMES)
print("Duration in seconds:", round(FRAMES / FPS, 2))

Frames per second: 25.0
Resolution: 1920 x 1080
Total frames: 250
Duration in seconds: 10.0


### Task 1.1: Uncompressed bandwidth math (5 points)

An uncompressed color frame uses about 3 bytes per pixel, one byte each for red,
green, and blue.

Fill in the TODO below so the cell prints the uncompressed data rate of this video in
megabits per second. Use the variables that were just printed above.

The formula is: width times height times 3 bytes times frames per second, then
multiply by 8 to convert bytes to bits, then divide by 1 million to get megabits.

In [8]:
mbps = WIDTH * HEIGHT * 3 * FPS * 8 / 1_000_000

print("Uncompressed data rate:", round(mbps, 1), "megabits per second")
print("File size on disk:", round(os.path.getsize(VIDEO) * 8 / 1_000_000, 1), "megabits total")

Uncompressed data rate: 1244.2 megabits per second
File size on disk: 33.8 megabits total


### Task 1.2: Short answer (5 points)

Compare the uncompressed number you just calculated to the actual file size on disk.
In two or three sentences, explain why they are so different and why that difference
matters when a company puts fifty cameras in a warehouse.

**YOUR ANSWER HERE**

The uncompressed rate is 1244.2 megabits per second, but the whole 10-second file on disk is only 33.8 megabits, about a 370x reduction. H.264 gets that by storing only what changes between frames instead of every pixel of every frame, which works especially well here because the camera is static and most of the scene is unchanging pavement. At scale that gap decides whether a system is buildable: fifty cameras streaming raw would need roughly 62 gigabits per second, far past any normal warehouse network, while fifty compressed streams fit under 200 megabits per second.

---

### Optional: use your own video instead

If you want to use your own footage, upload it with the Files panel on the left side
of Colab, then change the VIDEO variable above to your filename and re-run Part 1.
Pick something with several people or vehicles moving around. Keep it under 30
seconds.

---
# Part 2: Detection Only, No Memory

First we run plain detection. YOLO looks at each frame completely on its own. It has
no idea that the frames are related. It has no memory.

Watch what the output looks like and notice what is missing.

In [9]:
model = YOLO("yolov8n.pt")   # the small, fast model, good enough for this lab

# Detect people only. In the COCO dataset, class 0 is "person".
writer = cv2.VideoWriter("detect_only.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))

for result in model.predict(source=VIDEO, classes=[0], stream=True, verbose=False):
    writer.write(result.plot())

writer.release()
print("Saved detect_only.mp4")

Saved detect_only.mp4


In [10]:
show_video("detect_only.mp4")

### Task 2.1: Short answer (10 points)

Watch the video above. Pick one person near the middle of the frame and follow them
with your eyes.

Answer both questions in two to four sentences total:

1. What information does each box give you?

Each box gives the class label ("person"), the object's location and size in that frame via the box coordinates, and a confidence score for how certain the model is. All of this applies to a single frame only.

2. Looking only at what is drawn on the screen, could a computer program tell that the
person in frame 1 is the same person in frame 100? Explain why or why not.

No. Detection runs independently on each frame, and every box carries the same generic "person" label with no unique ID linking it across frames. Matching identities over time requires a separate tracking step.

**YOUR ANSWER HERE**

---
# Part 3: Add a Tracker and Get Identity

Now we change exactly one thing. Instead of `model.predict`, we call `model.track`
and hand it a tracker configuration. Everything else stays the same.

`bytetrack.yaml` is ByteTrack, the 2022 tracker from the lecture. `persist=True`
tells Ultralytics to carry track state from one frame to the next instead of starting
fresh each time.

This is the tracking by detection paradigm in practice. The detector finds. The
tracker connects.

In [11]:
model = YOLO("yolov8n.pt")   # fresh model so no track state carries over

writer = cv2.VideoWriter("track_bytetrack.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))

bytetrack_ids = collections.Counter()   # how many frames each ID survived

for result in model.track(source=VIDEO, classes=[0], tracker="bytetrack.yaml",
                          persist=True, stream=True, verbose=False):
    writer.write(result.plot())
    if result.boxes.id is not None:
        for track_id in result.boxes.id.int().tolist():
            bytetrack_ids[track_id] += 1

writer.release()
print("Saved track_bytetrack.mp4")
print("Total unique IDs assigned by ByteTrack:", len(bytetrack_ids))

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 321ms
Prepared 1 package in 55ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Saved track_bytetrack.mp4
Total unique IDs assigned by ByteTrack: 79


In [12]:
show_video("track_bytetrack.mp4")

### Task 3.1: Short answer (10 points)

Compare this video to the detection only video from Part 2. Describe in your own
words the one thing that is new on the screen, and explain what it means for a
computer program that has to count how many different people walked through the
square.

**YOUR ANSWER HERE**

The new element is a persistent ID number on each box, which links a detection in one frame to the same person in later frames instead of treating every frame independently. This allows a program to count unique people by counting the distinct IDs rather than summing boxes per frame, which would massively overcount by re-counting the same person hundreds of times.

---
# Part 4: Count the IDs and Find the Failures

Here is where the lab gets interesting. A tracker that works perfectly assigns
exactly one ID per real object. A tracker that struggles assigns extra IDs.

The cell below sorts every track by how many frames it survived. A track that lived
for two or three frames is almost never a real person. It is either a detector false
positive, which the lecture called a phantom track, or it is a fragment of a real
person whose identity got dropped and re-created.

In [13]:
print(f"{'Track ID':>10}  {'Frames alive':>13}  {'Seconds alive':>14}")
print("-" * 42)
for track_id, count in bytetrack_ids.most_common():
    print(f"{track_id:>10}  {count:>13}  {count / FPS:>14.2f}")

short_tracks = [t for t, n in bytetrack_ids.items() if n < FPS]   # alive under 1 second
print()
print("Total unique IDs:", len(bytetrack_ids))
print("IDs that lived less than one second:", len(short_tracks))

  Track ID   Frames alive   Seconds alive
------------------------------------------
         6            250           10.00
        12            250           10.00
        15            250           10.00
        31            250           10.00
        34            250           10.00
        36            250           10.00
        29            249            9.96
        40            247            9.88
         3            245            9.80
        11            241            9.64
         7            239            9.56
        16            239            9.56
        13            235            9.40
        26            227            9.08
        20            212            8.48
        21            212            8.48
        35            207            8.28
        82            207            8.28
       100            199            7.96
       101            198            7.92
       105            197            7.88
        69            187        

### Task 4.1: Count the real people (10 points)

Scrub through `track_bytetrack.mp4` above and count by hand how many distinct real
people appear in the 10 second clip. Take your time. Use the video controls to pause.

Fill in your count below and run the cell.

In [15]:
# TODO: replace 0 with the number of real distinct people you counted by hand
my_human_count = 41

tracker_count = len(bytetrack_ids)
print("People I counted by hand:", my_human_count)
print("IDs the tracker created:", tracker_count)
print("Extra IDs the tracker created:", tracker_count - my_human_count)

People I counted by hand: 41
IDs the tracker created: 79
Extra IDs the tracker created: 38


### Task 4.2: Diagnose the failures (15 points)

The lecture listed five tracking failure modes:

1. ID switches between similar looking objects
2. Lost tracks during long occlusions
3. Phantom tracks from detector false positives
4. Identity drift when objects change appearance
5. Cascading failures in crowded scenes

Watch your tracked video again and pick **two** of these five that you can actually
see happening in your output. For each one, write:

- The name of the failure mode
- Roughly when it happens, for example around second 4
- What you saw on screen that made you pick that mode

Do not guess. If you cannot see it, pick a different one. Being able to point at the
Screen and name the failure; this task is testing the entire skill.

**YOUR ANSWER HERE**

Failure 1: Lost tracks during long occlusions

I noticed a failure around second 3: two people in the upper-left group pass each other, and the person labeled as 11 changes to 112 before changing back to 11 again. The tracker created 79 IDs, but I was only able to count 41 after rewatching the video three times. This could be the reason for ID inflation. IDs as high as 216 on a 10-second clip mean tracks are being fragmented and re-spawned rather than one ID per person.

Failure 2: Phantom tracks from detector false positives

The confidence score also drops when it is hard to determine whether it is a person, as people bypass and overlap each other or walk farther away from the frame. The IDs also tend to change or vanish for a quick second before returning to their previous ID.



---
# Part 5: Compare Two Trackers

ByteTrack leans heavily on motion cues and on associating every detection box,
including the low confidence ones. BoT-SORT adds camera motion compensation and a
stronger appearance component.

Run BoT-SORT on the exact same video and compare.

In [16]:
model = YOLO("yolov8n.pt")   # fresh model again

writer = cv2.VideoWriter("track_botsort.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"), FPS, (WIDTH, HEIGHT))

botsort_ids = collections.Counter()

for result in model.track(source=VIDEO, classes=[0], tracker="botsort.yaml",
                          persist=True, stream=True, verbose=False):
    writer.write(result.plot())
    if result.boxes.id is not None:
        for track_id in result.boxes.id.int().tolist():
            botsort_ids[track_id] += 1

writer.release()
print("Saved track_botsort.mp4")

Saved track_botsort.mp4


In [17]:
show_video("track_botsort.mp4")

In [18]:
print(f"{'Tracker':<12} {'Unique IDs':>11} {'Short tracks':>14} {'Longest track (sec)':>21}")
print("-" * 62)
for name, ids in [("ByteTrack", bytetrack_ids), ("BoT-SORT", botsort_ids)]:
    short = len([t for t, n in ids.items() if n < FPS])
    longest = max(ids.values()) / FPS if ids else 0
    print(f"{name:<12} {len(ids):>11} {short:>14} {longest:>21.2f}")

print()
print("Your hand count of real people:", my_human_count)

Tracker       Unique IDs   Short tracks   Longest track (sec)
--------------------------------------------------------------
ByteTrack             79             24                 10.00
BoT-SORT              74             21                 10.00

Your hand count of real people: 41


### Task 5.1: Short answer (15 points)

Answer all three questions. Two to five sentences total is fine.

1. Which tracker produced a unique ID count closer to your hand count?
2. Which tracker held its longest track for more seconds?
3. The lecture said motion cues are cheap but fail when objects stop or change
direction, while appearance cues survive occlusion but cost extra compute. Using that
idea, explain why the two trackers gave different numbers on this specific video.

If both trackers gave you nearly identical numbers, say so, and explain what about
this particular clip made the choice of tracker not matter much.

**YOUR ANSWER HERE**

BoT-SORT was slightly closer to my hand count of 41, with 74 unique IDs versus ByteTrack's 79, and both held their longest track for the full 10 seconds. The Numbers are close because the camera is static, and pedestrians walk at a steady pace and collide with each other. BoT-SORT's appearance re-ID only recovered a few extra tracks after occlusion.

---
# Part 6: Apply It to a Houston Workplace

No code in this part. This is the part that shows up in your final project and in job
interviews.

Pick one specific Houston workplace task that could use video tracking. Be concrete.
Some starting points, though you are welcome to pick your own:

- Container movement at the Port of Houston
- Forklift and pedestrian separation in a distribution warehouse
- Patient fall detection in a Texas Medical Center hallway
- Personal protective equipment compliance on a refinery site
- Checkout and shrink monitoring in a grocery store
- Intersection traffic counting for the city

### Task 6.1: Define the task (10 points)

Describe your chosen task in three to five sentences. Say what the camera sees, what
objects get tracked, and what decision or alert the system produces. Name whether it
needs to run online or offline, and say why.

**YOUR ANSWER HERE**

A ceiling-mounted camera watches a shared aisle in a distribution warehouse, tracking two classes like forklift and person with persistent IDs so it can measure distance and closing speed between them across frames. When a pedestrian enters a forklift's path within a set threshold, the system sounds an aisle alarm and logs the near-miss with a timestamp and clip. It must run online, since the value is warning both parties before impact; an offline report would only document a collision that already happened.

### Task 6.2: Name the failure that would hurt most (10 points)

Of the five failure modes, which one would do the most damage in your specific task,
and what would the real world consequence be? Be specific about the consequence. An
ID switch in a port means a container goes to the wrong ship. What does it mean in
your task?

**YOUR ANSWER HERE**

Lost tracks during long occlusions would do the most damage. If a worker on foot walks behind a pallet rack or a stack of cartons and the tracker drops their ID, the system stops measuring distance between them and the approaching forklift. The alarm never fires at the exact moment the pedestrian is invisible to the driver, too. The real-world consequence is a struck-by injury or fatality in the aisle, plus an OSHA recordable and a system that safety staff stop trusting.

### Task 6.3: Privacy and ethics (10 points)

Answer all four:

1. Who is being recorded, and do they know
2. How long would you retain the video and the tracking data, and why that long
3. Who inside the organization gets access to it
4. Name one design change you would make to reduce privacy risk while still doing the
job. For example, discarding raw frames and keeping only counts

**YOUR ANSWER HERE**

Warehouse employees, forklift operators, and any contractors or visitors in the aisle. They should know, via signage at every entrance and coverage in onboarding and the employee handbook, with the union or employee reps consulted before deployment.
Raw video for 30 days, long enough to investigate a near-miss or complete a safety review, then auto-deleted. Anonymized near-miss counts and zone statistics are kept for a year to spot recurring hazard patterns.
Only the EHS/safety team and a named IT administrator, with role-based access and an audit log of every retrieval. Explicitly not line supervisors or HR, so the footage cannot be repurposed for productivity monitoring or discipline.
Discard raw frames after the real-time alert and store only bounding-box coordinates, IDs, and timestamps by default, pulling video only when a near-miss threshold is crossed and blurring faces on anything retained. This keeps the collision-avoidance function fully intact, since it runs on geometry rather than identity.

---
# Grading

| Task | What it covers | Points |
|---|---|---|
| 1.1 | Bandwidth calculation | 5 |
| 1.2 | Compression and scale reasoning | 5 |
| 2.1 | Detection has no identity | 10 |
| 3.1 | What tracking adds | 10 |
| 4.1 | Hand count versus tracker count | 10 |
| 4.2 | Diagnose two failure modes | 15 |
| 5.1 | Compare trackers, motion versus appearance | 15 |
| 6.1 | Define a Houston workplace task | 10 |
| 6.2 | Consequence of the worst failure | 10 |
| 6.3 | Privacy and ethics | 10 |
| **Total** | | **100** |

Full credit on the written tasks requires that you refer to something you actually
saw in your own output. Answers that only restate the lecture will earn partial
credit.

# Submission Checklist

Before you submit, confirm all of the following:

1. Every code cell has been run and shows output
2. All three videos play in the notebook
3. Every **YOUR ANSWER HERE** has been replaced with your own writing
4. Both `# TODO` lines have real values in them
5. Saved as `ITAI1378_Lab10_YourLastName.ipynb`

# If Something Breaks

**The install cell fails or times out.** Run it again. Colab occasionally drops a
package download.

**A video will not display.** Make sure you ran the `show_video` helper cell in Part
0. If the file exists but stays blank, run `!ls -la *.mp4` in a new cell and confirm
the file size is not zero.

**Everything is very slow.** You are on CPU. Go to Runtime, Change runtime type, T4
GPU, Save, then run all cells again from the top.

**The runtime disconnected and my variables are gone.** Colab clears memory when it
disconnects. Use Runtime, then Run all, and it will rebuild everything.

**I want to go further.** Try `yolov8s.pt` instead of `yolov8n.pt` and see whether a
larger detector changes your ID counts. That is the single fastest way to see why
tracking quality depends so much on detection quality.